## Load Built-in Dataset

In [3]:
from surprise import Dataset, Reader

file_path = r"ml-1m\ratings.dat"

reader = Reader(line_format='user item rating timestamp', 
                sep="::")

data = Dataset.load_from_file(file_path=file_path,
                              reader=reader)

type(data)

surprise.dataset.DatasetAutoFolds

## Peeking Inside the Dataset


In [ ]:
import pandas as pd

df = pd.read_csv(
    filepath_or_buffer=file_path,
    sep="::",
    names=["user_id", "movie_id", "rating", "timestamp"],   # custom column names
    engine='python'                                         # needed to handle the "::" separator properly
)

print(df.head(5))
print(f"The dataset has {df.shape[0]:,} ratings and {df.shape[1]} columns")

   user_id  movie_id  rating  timestamp
0        1      1193       5  978300760
1        1       661       3  978302109
2        1       914       3  978301968
3        1      3408       4  978300275
4        1      2355       5  978824291
The dataset has 1,000,209 ratings and 4 columns


## SVD Model + RandomizedSearchCV

In [ ]:
from surprise import SVD
from surprise.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_grid_svd = {
    "n_factors": randint(80, 150),      # Number of latent factors (hidden features)
    "n_epochs": randint(15, 40),        # Training iterations
    "lr_all": uniform(0.003, 0.008),    # Learning rate
    "reg_all": uniform(0.03, 0.1),      # Regularization
    "biased": [True, False]             # True = add user/movie bias offsets (avg rating tendencies), False = use only latent factors
}

rs_svd = RandomizedSearchCV(algo_class=SVD,
                            param_distributions=param_grid_svd,
                            n_iter=15,                           # Try 15 random combos
                            cv=3,                                   
                            measures=["rmse", "mae"],
                            random_state=42)

rs_svd.fit(data=data)

## SVD Model + RandomizedSearchCV

In [ ]:
from surprise import SVD
from surprise.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_grid_svd = {
    "n_factors": randint(80, 150),      # Number of latent factors (hidden features)
    "n_epochs": randint(15, 40),        # Training iterations
    "lr_all": uniform(0.003, 0.008),    # Learning rate
    "reg_all": uniform(0.03, 0.1),      # Regularization
    "biased": [True, False]             # True = add user/movie bias offsets (avg rating tendencies), False = use only latent factors
}

rs_svd = RandomizedSearchCV(algo_class=SVD,
                            param_distributions=param_grid_svd,
                            n_iter=15,                           # Try 15 random combos
                            cv=3,                                   
                            measures=["rmse", "mae"],
                            random_state=42)

rs_svd.fit(data=data)

## Initialize SVD With Best Hyperparameters From Randomized Search


In [ ]:
from surprise.model_selection import train_test_split

best_svd = SVD(**rs_svd.best_params['rmse'])

trainset, testset = train_test_split(data=data,
                                     test_size=0.2,
                                     random_state=42)

best_svd.fit(trainset)

In [ ]:
from surprise import accuracy

svd_predictions = best_svd.test(testset)
svd_rmse = accuracy.rmse(predictions=svd_predictions)

RMSE: 0.8549


In [ ]:
from surprise.model_selection import train_test_split

best_svd = SVD(**rs_svd.best_params['rmse'])

trainset, testset = train_test_split(data=data,
                                     test_size=0.2,
                                     random_state=42)

best_svd.fit(trainset)

In [8]:
from surprise import accuracy

svd_predictions = best_svd.test(testset)
svd_rmse = accuracy.rmse(predictions=svd_predictions)

RMSE: 0.8549


## KNNBaseline

In [17]:
from surprise.model_selection import RandomizedSearchCV
from surprise import KNNBaseline

param_grid_knn = {
    'k': [30, 40, 50],
    'sim_options': {
        'name': ['cosine', 'msd'],
        'user_based': [True, False],
        'min_support': [3, 5]
    }
}

rs_knn = RandomizedSearchCV(algo_class=KNNBaseline,
                            param_distributions=param_grid_knn,
                            n_iter=5,
                            cv=2,
                            measures=["rmse", "mae"],
                            random_state=42
                            )

rs_knn.fit(data=data)

Estimating biases using als...
Computing the cosine similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the cosine similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the cosine similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the cosine similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the cosine similarity matrix...
Done computing similarity matrix.


KeyboardInterrupt: 

In [ ]:
best_knn = KNNBaseline(**rs_knn.best_params['rmse'])
best_knn.fit(trainset=trainset)

In [ ]:
knn_predictions = best_knn.test(testset=testset)
knn_rmse = accuracy.rmse(predictions=knn_predictions) 